In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

import shap

In [2]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b", use_fast=True)
model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b").cuda()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": True,
    "max_new_tokens": 1,
    "temperature": 1,
    "top_k": 50,
    "no_repeat_ngram_size": 2,
}

In [4]:
# s = ["Mexico:peso :: US:", "Answer in one word: Leo gives Miu 3 apples. Miu eats 1 apple and throws away 1 apples. How many does Miu have left?"]
s = ['Fact: The capital of the state containing Dallas is']

In [5]:
explainer = shap.Explainer(model, tokenizer)
shap_values = explainer(s)

In [6]:
print(shap_values)

.values =
array([[[-3.50681455e-06],
        [ 2.76400235e+00],
        [-8.72354661e-01],
        [ 2.00342429e+00],
        [ 4.72185472e+00],
        [ 1.18334559e+00],
        [ 1.11659276e-01],
        [ 2.30677287e+00],
        [ 9.67523081e-01],
        [ 5.83311160e+00],
        [ 2.01544604e+00]]])

.base_values =
array([[-21.22565525]])

.data =
(array(['', 'Fact', ':', ' The', ' capital', ' of', ' the', ' state',
       ' containing', ' Dallas', ' is'], dtype=object),)


In [7]:
shap.plots.text(shap_values)

In [10]:
from scipy.special import softmax

vals = shap_values.values
if vals.shape[-1] == 1:
    vals = vals.squeeze(-1)

token_probs = softmax(vals, axis=1)
# print the array (removed invalid 'split' keyword)
print(token_probs)
# Optionally, print the first row with comma-separated formatted probabilities:
# print(", ".join(f"{p:.6f}" for p in token_probs[0]))
# tokens = shap_values.data[0]
# for tok, p in zip(tokens, token_probs[0]):
#     print(f"  {tok!r}: {p:.4f}")

[[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
  0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]]


In [9]:
tokens = shap_values.data[0]
for tok, p in zip(tokens, token_probs[0]):
    print(f"  {tok!r}: {p:.4f}")

  '': 0.0020
  'Fact': 0.0315
  ':': 0.0008
  ' The': 0.0147
  ' capital': 0.2234
  ' of': 0.0065
  ' the': 0.0022
  ' state': 0.0200
  ' containing': 0.0052
  ' Dallas': 0.6787
  ' is': 0.0149
